In [1]:
!pip install -q "protobuf<4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.1 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12.4.

In [2]:
import json
import torch
import random

from tqdm import tqdm
from pathlib import Path
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer, logging

logging.set_verbosity_error()

In [3]:
def load_jsonl(file_path, max_samples=1000):
    with open(file_path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f]
    return data[:max_samples]

In [4]:
NUM_EXAMPLES = 200

gsm8k_test = load_jsonl("/kaggle/input/math-reasoning-dataset/data/gsm8k/test.jsonl", NUM_EXAMPLES)
math_test = load_jsonl("/kaggle/input/math-reasoning-dataset/data/mathematics/test.jsonl", NUM_EXAMPLES)

In [5]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

2025-12-25 23:03:34.550643: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766703814.711742      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766703814.757158      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766703815.146232      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766703815.146261      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766703815.146263      24 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
COT_PROMPT_TEMPLATE = """You are a highly precise reasoning assistant. Solve the problem step by step.

Problem: {question}

INSTRUCTIONS (READ CAREFULLY):
1. Show your step-by-step reasoning in **numbered steps**, e.g.:
   Step 1: ...
   Step 2: ...
   Final answer: ...
3. THE FINAL ANSWER MUST BE STRICTLY AN INTEGER NUMBER. NO UNITS, NO WORDS, NO SYMBOLS, NO LATEX.
4. THE FINAL ANSWER MUST BE OUTPUT IN THIS EXACT FORMAT:
   Final answer: #### <final_number>
   Replace <final_number> with only the INTEGER NUMERIC ANSWER WITHOUT ANITHING ELSE.
5. DO NOT INCLUDE ANYTHING ELSE AFTER '####'.
6. IF YOU CANNOT CALCULATE, OUTPUT ONLY '#### NO ANSWER' AS PLACEHOLDER.

BEGIN STEP-BY-STEP REASONING:
"""

AGGREGATION_PROMPT_TEMPLATE = """You are a highly precise reasoning assistant.

You are given a math problem and several candidate solutions. Some candidates may be incorrect, incomplete, or contain reasoning errors.
Your task is to aggregate the useful ideas from the candidates and produce a single, high-quality solution.

Problem:
{question}

Candidate solutions:
{candidates}

INSTRUCTIONS (READ CAREFULLY):
1. Analyze all candidate solutions step by step.
2. Identify correct, useful, or partially correct reasoning steps.
3. If candidates disagree, **select the logically correct path** and discard incorrect reasoning.
4. Combine the selected reasoning into **one coherent, numbered step-by-step solution**, e.g.:
   Step 1: ...
   Step 2: ...
   Final answer: ...
5. If **all candidate solutions are incorrect or inconsistent**, abandon them and solve the problem using a correct alternative strategy.
6. THE FINAL ANSWER MUST BE STRICTLY AN INTEGER NUMBER.
   - NO UNITS
   - NO WORDS
   - NO SYMBOLS
   - NO LATEX
7. THE FINAL ANSWER MUST BE OUTPUT IN THIS EXACT FORMAT:
   Final answer: #### <final_number>
8. Replace <final_number> with ONLY the integer numeric answer.
9. DO NOT INCLUDE ANYTHING ELSE AFTER '####'.

BEGIN AGGREGATED STEP-BY-STEP REASONING:
"""

In [7]:
def majority_vote(population):
    answers = []
    for content in population:
        if "####" in content:
            ans = content.split("####")[-1].strip()
            answers.append(ans)

    if not answers:
        return ""

    counter = Counter(answers)
    return counter.most_common(1)[0][0]

In [8]:
def generate_populations_from_prompts(prompts, num_questions, question_indices, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt} for prompt in prompts]
    texts = [tokenizer.apply_chat_template([m], tokenize=False, add_generation_prompt=True) for m in messages]

    model_inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    populations = [[] for _ in range(num_questions)]
    for i, q_idx in enumerate(question_indices):
        output_ids = generated_ids[i][len(model_inputs.input_ids[i]):].tolist()
        content = tokenizer.decode(output_ids, skip_special_tokens=True)
        populations[q_idx].append(content)
    return populations

In [9]:
def rsa_batch_with_batchsize(questions, N=4, K=2, T=2):
    num_questions = len(questions)

    prompts = []
    question_indices = []
    for q_idx, q in enumerate(questions):
        for _ in range(N):
            prompts.append(COT_PROMPT_TEMPLATE.format(question=q))
            question_indices.append(q_idx)

    populations = generate_populations_from_prompts(prompts, num_questions, question_indices)

    for _ in range(T):
        agg_prompts = []
    
        for i, q_idx in enumerate(question_indices):
            chosens = random.sample(populations[q_idx], K)
            candidates_text = "\n\n".join([f"CANDIDATE #{i+1}:\n{c}" for i, c in enumerate(chosens)])

            agg_prompt = AGGREGATION_PROMPT_TEMPLATE.format(
                question=questions[q_idx],
                candidates=candidates_text
            )
            agg_prompts.append(agg_prompt)
    
        populations = generate_populations_from_prompts(agg_prompts, num_questions, question_indices)

    final_answers = []
    for pop in populations:
        final_answer = majority_vote(pop)
        final_answers.append(final_answer)
    
    return final_answers

In [10]:
def compute_accuracy(dataset, N=4, K=2, T=2, batch_size=40):
    assert batch_size % N == 0
    num_questions = batch_size // N
    
    correct = 0
    
    questions = [item["question"] for item in dataset]
    answers = [item["answer"] for item in dataset]

    for i in tqdm(range(0, len(questions), num_questions)):
        batch_q = questions[i:i + num_questions]
        batch_a = answers[i:i + num_questions]
        batch_p = rsa_batch_with_batchsize(batch_q, N, K, T)
        for pred, real in zip(batch_p, batch_a):
            if pred == real:
                correct += 1

    return correct / len(dataset)

In [11]:
gsm8k_base_acc = compute_accuracy(gsm8k_test, N=1, K=0, T=0)

print(f"GSM8K baseline accuracy: {gsm8k_base_acc:.1%}")

100%|██████████| 5/5 [05:15<00:00, 63.05s/it]

GSM8K baseline accuracy: 59.0%


In [12]:
gsm8k_rsa_acc = compute_accuracy(gsm8k_test, N=4, K=2, T=2)

print(f"GSM8K RSA accuracy: {gsm8k_rsa_acc:.1%}")

100%|██████████| 20/20 [1:46:22<00:00, 319.14s/it]

GSM8K RSA accuracy: 69.5%


In [13]:
math_base_acc = compute_accuracy(math_test, N=1, K=0, T=0)

print(f"Mathematics baseline accuracy: {math_base_acc:.1%}")

100%|██████████| 5/5 [05:15<00:00, 63.06s/it]

Mathematics baseline accuracy: 36.0%


In [14]:
math_rsa_acc = compute_accuracy(math_test, N=4, K=2, T=2)

print(f"Mathematics RSA accuracy: {math_rsa_acc:.1%}")

100%|██████████| 20/20 [1:58:10<00:00, 354.51s/it]

Mathematics RSA accuracy: 53.5%
